In [2]:
import pandas as pd
import numpy as np

In [12]:
metrics = pd.read_csv(
    "/data/dmytro/Experiments/ExtendedProportions/pseudobulk/LookupClassifier_SoftLabels_WITHOUTPOOLING/pseudobulk/fitted_deconvolvers_unifrorm_multinomial_all_top156_features/callibration//calibration_summary.csv"
)

In [14]:
deconv_map = {"xgb": "XGB", "mlp": "MLP", "swn": "SWN", "nnls": "NNLS", "psls": "PSLS"}
calib_map = {
    "uncalibrated": "None",
    "linear_clip0_normalize": "Lin.\\ clip0+norm",
    "linear_simplex_projection": "Lin.\\ simplex",
    "vector_scaling": "Vec.\\ scaling",
}

deconv_order = ["xgb", "mlp", "swn", "nnls", "psls"]
# deconv_order = ["xgb", "swn",  "psls"]
calib_order = list(calib_map.keys())
n_calib = len(calib_order)
n_total = len(deconv_order) * n_calib

lines = []
first_row = True
for i, dec in enumerate(deconv_order):
    for j, cal in enumerate(calib_order):
        row = metrics[
            (metrics["deconvolver"] == dec) & (metrics["calibration_method"] == cal)
        ].iloc[0]

        r2 = f"{row['overall_r2'] * 100:.2f}"
        loa = f"[{row['loa_lower']*1e2:.2f}, {row['loa_upper']*1e2:.2f}]"
        loa_worst = f"[{row['worst_class_loa_lower']*1e2:.2f}, {row['worst_class_loa_upper']*1e2:.2f}]"
        mae = f"{row['mae']*1e3:.2f}"
        mse = f"{row['mse']*1e4:.2f}"
        kl = f"{row['kl']*1e2:.2f}"

        cal_label = calib_map[cal]

        if first_row:
            lines.append(
                f"                             & \\multirow{{{n_total}}}{{*}}{{Uniform}}"
            )
            lines.append(
                f"                             & \\multirow{{{n_calib}}}{{*}}{{{deconv_map[dec]}}}"
                f"              & {cal_label:<25s} & {r2:<17s} & {loa:<19s} & {loa_worst:<23s} & {mae:<12s} & {mse:<12s} & {kl} \\\\"
            )
            first_row = False
        elif j == 0:
            lines.append(
                f"                                                                 "
                f"& \\multirow{{{n_calib}}}{{*}}{{{deconv_map[dec]}}} & {cal_label:<25s} & {r2:<17s} & {loa:<19s} & {loa_worst:<23s} & {mae:<12s} & {mse:<12s} & {kl} \\\\"
            )
        else:
            lines.append(
                f"                                                                 "
                f"&                       & {cal_label:<25s} & {r2:<17s} & {loa:<19s} & {loa_worst:<23s} & {mae:<12s} & {mse:<12s} & {kl} \\\\"
            )

    if i < len(deconv_order) - 1:
        lines.append("        \\cmidrule(l){2-9}")

print("\n".join(lines))

                             & \multirow{20}{*}{Uniform}
                             & \multirow{4}{*}{XGB}              & None                      & 75.99             & [-8.75, 8.75]       & [-18.74, 17.67]         & 15.16        & 19.93        & 49.14 \\
                                                                 &                       & Lin.\ clip0+norm          & 83.57             & [-7.24, 7.24]       & [-17.74, 19.56]         & 10.93        & 13.64        & 35.16 \\
                                                                 &                       & Lin.\ simplex             & 85.79             & [-6.73, 6.73]       & [-18.27, 19.32]         & 9.22         & 11.80        & 41.47 \\
                                                                 &                       & Vec.\ scaling             & 86.58             & [-6.54, 6.54]       & [-17.42, 19.80]         & 10.25        & 11.14        & 27.77 \\
        \cmidrule(l){2-9}
                                     